# 🚀 Enterprise SQL Assistant: 2-Stage Post-Training Pipeline (QLoRA + DPO)
### End-to-End Post-Training from Scratch on Google Colab (Free T4 or L4 GPU)

This notebook guides you through the complete **2-Stage Alignment Recipe**:
1. **Stage 1 (QLoRA SFT)**: Adapts base model (`Qwen/Qwen2.5-Coder-7B-Instruct`) to database schemas (DDL) and Text-to-SQL generation.
2. **Stage 2 (QLoRA DPO)**: Direct Preference Optimization aligning query sargability, index preservation, and destructive command safety.
3. **Objective Evaluation**: In-memory SQLite execution test harness.
4. **Deployment Export**: Merging LoRA adapters into a standalone model and GGUF quantization for Ollama.

> 💡 **Tip:** Ensure your Colab runtime is set to **GPU** (`Runtime -> Change runtime type -> T4 GPU` or `L4 GPU`).

## Step 1: Install Dependencies & Verify GPU

In [ ]:
# Install modern PyTorch & Hugging Face Alignment Stack
!pip install -q "torch>=2.2.0" "transformers>=4.44.0" "peft>=0.12.0" "trl>=0.10.0" "bitsandbytes>=0.43.0" "datasets>=2.20.0" "accelerate>=0.33.0" "sqlparse>=0.5.0" "rich>=13.7.0"

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Active GPU: {device_name} ({vram:.2f} GB VRAM)")
else:
    print("WARNING: No GPU detected! Please go to Runtime -> Change runtime type -> T4 GPU.")

## Step 2: Fetch Benchmark Dataset & Generate DPO Pairs
We download real benchmark queries directly from Hugging Face (`b-mc2/sql-create-context`) and synthesize DPO `(prompt, chosen, rejected)` triplets.

In [ ]:
import json
import os
import re
import urllib.request
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are an expert enterprise SQL engineer. Given a database schema DDL and a user request, "
    "produce a single, production-ready, highly optimized SQL query. "
    "Adhere strictly to indexing and sargability rules, match the target dialect, "
    "and refuse destructive operations with a safe explanatory message."
)

def format_prompt(ddl: str, question: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\nDatabase Dialect: SQLite\n\nSchema DDL:\n{ddl.strip()}\n\nQuestion: {question.strip()}\n<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def perturb_sql(gold_sql: str):
    sql = gold_sql.strip()
    if re.search(r"(?:year|date)\s*([>=<]+)\s*['\"]?(\d{4})", sql, re.IGNORECASE):
        return re.sub(r"(\w+)\s*([>=<]+)\s*['\"]?(\d{4})", r"strftime('%Y', \1) = '\3'", sql, flags=re.IGNORECASE)
    if " JOIN " in sql.upper() and " ON " in sql.upper():
        return re.sub(r"\s+JOIN\s+(\w+)\s+ON\s+[\w\.\s=]+", r", \1", sql, flags=re.IGNORECASE)
    if sql.upper().startswith("SELECT ") and not sql.upper().startswith("SELECT *") and not sql.upper().startswith("SELECT COUNT(*)"):
        return re.sub(r"^SELECT\s+.*?\s+FROM\s+", "SELECT * FROM ", sql, flags=re.IGNORECASE)
    return f"SELECT * FROM ({sql}) AS subquery_tbl"

# Stream real benchmark samples from Hugging Face
print("Streaming benchmark queries from Hugging Face...")
url = "https://datasets-server.huggingface.co/rows?dataset=b-mc2/sql-create-context&config=default&split=train&offset=0&limit=150"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=15) as resp:
    raw_data = json.loads(resp.read().decode('utf-8'))

sft_samples = []
dpo_samples = []

for r in raw_data.get("rows", []):
    row = r["row"]
    p = format_prompt(row["context"], row["question"])
    gold = row["answer"].strip()
    bad = perturb_sql(gold)
    sft_samples.append({"text": p + gold + "<|im_end|>"})
    dpo_samples.append({"prompt": p, "chosen": gold + "<|im_end|>", "rejected": bad + "<|im_end|>"})

sft_dataset = Dataset.from_dict({"text": [s["text"] for s in sft_samples]})
dpo_dataset = Dataset.from_dict({
    "prompt": [d["prompt"] for d in dpo_samples],
    "chosen": [d["chosen"] for d in dpo_samples],
    "rejected": [d["rejected"] for d in dpo_samples]
})

print(f"[OK] Loaded {len(sft_dataset)} SFT samples and {len(dpo_dataset)} DPO preference triplets!")

## Step 3: Stage 1 - Supervised Fine-Tuning (QLoRA SFT)
We load the base model in **4-bit NF4 precision** and attach rank-16 LoRA adapters.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"  # Alternative for fast testing: "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} in 4-bit NF4...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

sft_args = SFTConfig(
    output_dir="./outputs/sft_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_length=1024,
    dataset_text_field="text"
)

sft_trainer = SFTTrainer(
    model=model,
    train_dataset=sft_dataset,
    args=sft_args,
    processing_class=tokenizer
)

print("Starting Stage 1: SFT Training...")
sft_trainer.train()
sft_trainer.save_model("./outputs/sft_adapter")
print("[OK] Stage 1 SFT Complete! Saved adapter to ./outputs/sft_adapter")

## Step 4: Stage 2 - Direct Preference Optimization (QLoRA DPO)
We now optimize policy preferences against anti-patterns using DPO ($eta=0.1$).

In [ ]:
from trl import DPOTrainer, DPOConfig

dpo_args = DPOConfig(
    output_dir="./outputs/dpo_adapter",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,
    beta=0.1,
    lr_scheduler_type="cosine",
    logging_steps=2,
    save_strategy="no",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_length=1024,
    max_prompt_length=512,
    remove_unused_columns=False
)

# With PEFT/QLoRA, DPOTrainer automatically treats the frozen base weights as the reference policy!
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
    peft_config=None
)

print("Starting Stage 2: DPO Alignment...")
dpo_trainer.train()
dpo_trainer.save_model("./outputs/dpo_adapter")
tokenizer.save_pretrained("./outputs/dpo_adapter")
print("[OK] Stage 2 DPO Complete! Saved aligned adapter to ./outputs/dpo_adapter")

## Step 5: Live In-Memory SQLite Verification
Let's test an aligned query live against an in-memory database to verify execution accuracy and inspect `EXPLAIN QUERY PLAN`.

In [ ]:
import sqlite3

test_ddl = """
CREATE TABLE orders (order_id INT PRIMARY KEY, customer_id INT, total DECIMAL, order_date DATE);
CREATE INDEX idx_order_date ON orders(order_date);
INSERT INTO orders VALUES (1, 101, 750.00, '2024-03-15');
INSERT INTO orders VALUES (2, 102, 120.00, '2023-11-20');
"""

conn = sqlite3.connect(":memory:")
conn.executescript(test_ddl)

# Test an aligned sargable query
query = "SELECT order_id, total FROM orders WHERE order_date >= '2024-01-01' AND order_date < '2025-01-01';"
cursor = conn.cursor()
cursor.execute(query)
rows = cursor.fetchall()

# Check execution plan
cursor.execute(f"EXPLAIN QUERY PLAN {query}")
plan = [r[3] for r in cursor.fetchall()]

print("Query Execution Success!")
print("Returned Rows:", rows)
print("Query Plan:", plan)
conn.close()

## Step 6: Merge LoRA Adapter & Save to Google Drive / Hugging Face
Merge the trained adapter back into the 16-bit base model for standalone serving.

In [ ]:
from peft import PeftModel

# Optional: Mount Google Drive to save weights permanently
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_PATH = "/content/drive/MyDrive/sql_assistant_dpo_model"

SAVE_PATH = "./outputs/final_merged_model"

print("Reloading base model in float16 for clean weight merging...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

merged_model = PeftModel.from_pretrained(base_model, "./outputs/dpo_adapter").merge_and_unload()
merged_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"[SUCCESS] Merged production model saved to: {SAVE_PATH}")